In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# 1. Load your MIMIC-III dataset
# base_path = '/content/drive/My Drive/MIMIC_III_REAL/'
# df_mimic3 = pd.read_csv(base_path + 'MIMIC_III_CARDIAC_ML_READY.csv') # Adjust filename if necessary
base_path = '/content/drive/MyDrive/MIMIC_III_REAL/'

df_mimic3 = pd.read_csv(
    base_path + 'MIMIC_III_CARDIAC_ML_READY.csv'
)
# 2. Separate Target and Group Identifiers
# Note: Screens show 'hospital_expire_flag' or 'is_deceased'. We use hospital_expire_flag to match MIMIC-4
y_m3 = df_mimic3['hospital_expire_flag'].astype(int)
groups_m3 = df_mimic3['subject_id']

# 3. Identify and Isolate Features explicitly present in MIMIC-III screenshots
cols_to_exclude = ['subject_id', 'hadm_id', 'icd_code', 'icd_description', 'gender', 'hospital_expire_flag', 'is_deceased']
X_m3 = df_mimic3.drop(columns=[col for col in cols_to_exclude if col in df_mimic3.columns])

# Clean column names to match any alphanumeric stripping done previously
X_m3 = X_m3.rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

# Define your continuous clinical columns for scaling
continuous_cols = [
    'anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
    'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
    'vital_mean_spo2', 'vital_mean_temp'
]

# 4. Split using GroupKFold to maintain patient isolation
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X_m3, y_m3, groups=groups_m3))

X_train_m3 = X_m3.iloc[train_idx].copy()
X_test_m3 = X_m3.iloc[test_idx].copy()
y_train_m3 = y_m3.iloc[train_idx]
y_test_m3 = y_m3.iloc[test_idx]
groups_train_m3 = groups_m3.iloc[train_idx]

# 5. Scale continuous features (This ensures your vitals/age are normalized properly)
scaler = StandardScaler()
X_train_m3[continuous_cols] = scaler.fit_transform(X_train_m3[continuous_cols])
X_test_m3[continuous_cols] = scaler.transform(X_test_m3[continuous_cols])

print(f"MIMIC-III Feature matrix shape: {X_m3.shape}")

MIMIC-III Feature matrix shape: (36396, 88)


In [8]:
from sklearn.impute import SimpleImputer

# Handle NaNs quickly
imputer = SimpleImputer(strategy='median')
X_train_m3_clean = pd.DataFrame(imputer.fit_transform(X_train_m3), columns=X_train_m3.columns)
X_test_m3_clean = pd.DataFrame(imputer.transform(X_test_m3), columns=X_test_m3.columns)

param_distributions = {
    'C': np.logspace(-2, 1, 10),
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

cv_strategy = GroupKFold(n_splits=5)

# OPTIMIZED: Added tol=0.1 and lowered max_iter so SAGA converges significantly faster
base_lr = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight='balanced',
    max_iter=100,
    tol=0.1,
    random_state=42
)

print("Running fast RandomizedSearchCV over MIMIC-III features...")
lr_search_m3 = RandomizedSearchCV(
    estimator=base_lr,
    param_distributions=param_distributions,
    n_iter=8,
    scoring='roc_auc',
    cv=cv_strategy.split(X_train_m3_clean, y_train_m3, groups=groups_train_m3),
    random_state=42,
    n_jobs=1  # Changed from -1 to 1 to prevent Colab multiprocessing freeze
)

# Fit search on the clean data
lr_search_m3.fit(X_train_m3_clean, y_train_m3)

print("\n--- OPTIMIZATION SEARCH RESULTS COMPLETE (MIMIC-III) ---")
print(f"Best Parameters -> C: {lr_search_m3.best_params_['C']:.4f} | l1_ratio: {lr_search_m3.best_params_['l1_ratio']:.2f}")

Running fast RandomizedSearchCV over MIMIC-III features...

--- OPTIMIZATION SEARCH RESULTS COMPLETE (MIMIC-III) ---
Best Parameters -> C: 10.0000 | l1_ratio: 0.10


In [9]:
# 1. Predict labels using the optimized model
y_pred_m3 = lr_search_m3.best_estimator_.predict(X_test_m3_clean)

# 2. Print the evaluation report
print(classification_report(y_test_m3, y_pred_m3))

              precision    recall  f1-score   support

           0       0.96      0.81      0.88      6310
           1       0.40      0.80      0.53       970

    accuracy                           0.81      7280
   macro avg       0.68      0.81      0.71      7280
weighted avg       0.89      0.81      0.84      7280



In [10]:
print(f"Optimized Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_m3['y_true'], patient_res_m3['y_prob']):.4f}")

Optimized Patient-Level ROC-AUC Score: 0.9535
